In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
# Import PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

# Import Torchvision
import torchvision
import torchvision.transforms as transforms

# Import other utilities
import numpy as np
import matplotlib.pyplot as plt

# Check versions
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

# Check GPU
print("GPU Available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
GPU Available: True


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

Using device: cuda
GPU Name: Tesla T4


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

In [ ]:
train_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

val_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=test_transform
)

test_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

print("Dataset downloaded successfully!")

100%|██████████| 170M/170M [35:25<00:00, 80.2kB/s]


Dataset downloaded successfully!


In [ ]:
print("Training images:", len(train_data))
print("Validation dataset:", len(val_data))
print("Testing images:", len(test_data))

print("Classes:", train_data.classes)


Training images: 50000
Validation dataset: 50000
Testing images: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
!ls -lh ./data/cifar-10-batches-py/

total 178M
-rw-r--r-- 1 2156 1103 158 Mar 31  2009 batches.meta
-rw-r--r-- 1 2156 1103 30M Mar 31  2009 data_batch_1
-rw-r--r-- 1 2156 1103 30M Mar 31  2009 data_batch_2
-rw-r--r-- 1 2156 1103 30M Mar 31  2009 data_batch_3
-rw-r--r-- 1 2156 1103 30M Mar 31  2009 data_batch_4
-rw-r--r-- 1 2156 1103 30M Mar 31  2009 data_batch_5
-rw-r--r-- 1 2156 1103  88 Jun  4  2009 readme.html
-rw-r--r-- 1 2156 1103 30M Mar 31  2009 test_batch


In [ ]:
from torch.utils.data import Subset, DataLoader

# Set random seed for reproducibility
torch.manual_seed(42)

# Generate random indices
indices = torch.randperm(50000).tolist()

# Split indices
train_indices = indices[:45000]
val_indices = indices[45000:]

# Create subsets
train_subset = Subset(train_data, train_indices)
val_subset = Subset(val_data, val_indices)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Testing images:", len(test_data))

Training images: 45000
Validation images: 5000
Testing images: 10000


In [ ]:
batch_size = 128

train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully!")

DataLoaders created successfully!


In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Number of classes:", len(train_data.classes))

Image batch shape: torch.Size([128, 3, 32, 32])
Label batch shape: torch.Size([128])
Number of classes: 10


In [ ]:
import torch.nn as nn
from torchvision import models

In [ ]:
# Load ResNet-18 architecture
teacher_model = models.resnet18(weights=None)

# Modify first convolutional layer for CIFAR-10
teacher_model.conv1 = nn.Conv2d(
    3, 64,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False
)

# Remove max pooling layer
teacher_model.maxpool = nn.Identity()

# Modify final fully connected layer
teacher_model.fc = nn.Linear(512, 10)

# Move model to GPU
teacher_model = teacher_model.to(device)

print(teacher_model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

In [ ]:
# Take one batch of images
images, labels = next(iter(train_loader))

# Move images to GPU
images = images.to(device)

# Pass images through teacher model
outputs = teacher_model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([128, 3, 32, 32])
Output shape: torch.Size([128, 10])


In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.SGD(
    teacher_model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4
)

In [ ]:
num_epochs = 20

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[10, 15],
    gamma=0.1
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Epochs:", num_epochs)

Loss function: CrossEntropyLoss()
Optimizer: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.1
    lr: 0.1
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0.0005
)
Epochs: 20


In [ ]:
print("Teacher model:", type(teacher_model).__name__)
print("Device:", next(teacher_model.parameters()).device)
print("Total training images:", len(train_subset))
print("Batch size:", batch_size)
print("Number of epochs:", num_epochs)

Teacher model: ResNet
Device: cuda:0
Total training images: 45000
Batch size: 128
Number of epochs: 20


In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Calculate accuracy
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
def validate(model, val_loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_loss / total
    val_accuracy = 100 * correct / total

    return val_loss, val_accuracy

In [ ]:
best_val_accuracy = 0.0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        teacher_model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        teacher_model,
        val_loader,
        criterion,
        device
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.2f}%"
    )

    # Save best teacher model
    if val_acc > best_val_accuracy:

        best_val_accuracy = val_acc

        torch.save(
            teacher_model.state_dict(),
            "best_teacher.pth"
        )

print("Teacher training completed!")
print("Best validation accuracy:", best_val_accuracy)

Epoch [1/20] Train Loss: 1.9708 Train Acc: 30.22% Val Loss: 1.6307 Val Acc: 39.70%
Epoch [2/20] Train Loss: 1.4004 Train Acc: 48.40% Val Loss: 1.2306 Val Acc: 56.32%
Epoch [3/20] Train Loss: 1.1252 Train Acc: 59.53% Val Loss: 1.3981 Val Acc: 53.88%
Epoch [4/20] Train Loss: 0.9519 Train Acc: 66.44% Val Loss: 0.9823 Val Acc: 65.72%
Epoch [5/20] Train Loss: 0.8137 Train Acc: 71.42% Val Loss: 0.8520 Val Acc: 70.32%
Epoch [6/20] Train Loss: 0.7148 Train Acc: 75.14% Val Loss: 0.8257 Val Acc: 71.72%
Epoch [7/20] Train Loss: 0.6337 Train Acc: 77.90% Val Loss: 0.6390 Val Acc: 77.60%
Epoch [8/20] Train Loss: 0.5796 Train Acc: 80.10% Val Loss: 0.7690 Val Acc: 73.90%
Epoch [9/20] Train Loss: 0.5388 Train Acc: 81.37% Val Loss: 0.6373 Val Acc: 77.60%
Epoch [10/20] Train Loss: 0.5135 Train Acc: 82.32% Val Loss: 0.9536 Val Acc: 70.84%
Epoch [11/20] Train Loss: 0.3375 Train Acc: 88.55% Val Loss: 0.3344 Val Acc: 88.32%
Epoch [12/20] Train Loss: 0.2870 Train Acc: 90.28% Val Loss: 0.3264 Val Acc: 88.72%
E

In [ ]:
# Load the best saved teacher weights
teacher_model.load_state_dict(
    torch.load("best_teacher.pth", map_location=device, weights_only=True)
)

teacher_model.eval()

print("Best teacher model loaded successfully!")

Best teacher model loaded successfully!


In [ ]:
test_loss, test_accuracy = validate(
    teacher_model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.2917
Test Accuracy: 90.30%


In [ ]:
torch.save(
    teacher_model.state_dict(),
    "teacher_model.pth"
)

print("Teacher model saved successfully!")

Teacher model saved successfully!


In [ ]:
import os

print("Current folder:", os.getcwd())
print("Model in current folder:", os.path.exists("teacher_model.pth"))
print("Best checkpoint exists:", os.path.exists("best_teacher.pth"))

Current folder: /content
Model in current folder: True
Best checkpoint exists: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Knowledge_Distillation', exist_ok=True)

import shutil
shutil.copy2(
    "teacher_model.pth",
    "/content/drive/MyDrive/Knowledge_Distillation/teacher_model.pth"
)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/Knowledge_Distillation"
os.makedirs(project_path, exist_ok=True)

print("Folder ready!")

Folder ready!


In [ ]:
import torch

teacher_model.load_state_dict(
    torch.load("best_teacher.pth", map_location=device, weights_only=True)
)

torch.save(
    teacher_model.state_dict(),
    project_path + "/teacher_model.pth"
)

print("Teacher model saved to Google Drive!")

Teacher model saved to Google Drive!


In [ ]:
import os

print(os.listdir(project_path))

['teacher_model.pth']


In [ ]:
from google.colab import files

files.download(project_path + "/teacher_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
teacher_model.load_state_dict(
    torch.load("best_teacher.pth", map_location=device, weights_only=True)
)

teacher_model.eval()
print("Best teacher checkpoint loaded!")

Best teacher checkpoint loaded!


In [36]:
test_loss, test_accuracy = validate(
    teacher_model,
    test_loader,
    criterion,
    device
)

print(f"Teacher Test Loss: {test_loss:.4f}")
print(f"Teacher Test Accuracy: {test_accuracy:.2f}%")

Teacher Test Loss: 0.2917
Teacher Test Accuracy: 90.30%


In [37]:
import torch

teacher_model.load_state_dict(
    torch.load("best_teacher.pth", map_location=device, weights_only=True)
)

torch.save(
    teacher_model.state_dict(),
    "/content/drive/MyDrive/Knowledge_Distillation/teacher_model.pth"
)

print("Best teacher model saved!")

Best teacher model saved!


In [38]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/Knowledge_Distillation/teacher_model.pth"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>